# Zip / Unzip Notebook (ZIP & LZ4)

| Cell | Operation |
|------|-----------|
| **1** | Install & Imports |
| **2** | Helper functions (run once) |
| **3** | ▶ ZIP — Compress a single folder |
| **4** | ▶ ZIP — Compress every sub-folder inside a parent folder |
| **5** | ▶ ZIP — Extract a single `.zip` file |
| **6** | ▶ ZIP — Extract every `.zip` inside a folder |
| **7** | ▶ LZ4 — Compress a single folder (`.tar.lz4`) |
| **8** | ▶ LZ4 — Compress every sub-folder inside a parent folder |
| **9** | ▶ LZ4 — Extract a single `.tar.lz4` file |
| **10** | ▶ LZ4 — Extract every `.tar.lz4` inside a folder |
| **11** | ▶ TAR.GZ — Extract a single `.tar.gz` file |
| **12** | ▶ TAR.GZ — Extract every `.tar.gz` inside a folder |

In [1]:
%pip install lz4 tqdm --quiet

import os
import io
import sys
import shutil
import zipfile
import tarfile
import time
from pathlib import Path

import lz4.frame
from tqdm import tqdm

print("✓ Imports ready (zipfile, tarfile, lz4.frame, tqdm)")

Note: you may need to restart the kernel to use updated packages.
✓ Imports ready (zipfile, tarfile, lz4.frame, tqdm)


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper functions — run this cell once before any of the action cells below.
# ─────────────────────────────────────────────────────────────────────────────

def _human_size(num_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if num_bytes < 1024:
            return f"{num_bytes:,.2f} {unit}"
        num_bytes /= 1024
    return f"{num_bytes:,.2f} PB"


def _iter_files(folder: Path):
    for root, _dirs, files in os.walk(folder):
        for name in files:
            yield Path(root) / name


def _folder_size(folder: Path) -> int:
    return sum(f.stat().st_size for f in _iter_files(folder) if f.is_file())


# ── ZIP helpers ──────────────────────────────────────────────────────────────
def zip_folder(src_folder, out_zip=None, compression=zipfile.ZIP_DEFLATED) -> Path:
    """Compress a folder into a single .zip file (preserves folder structure)."""
    src = Path(src_folder).resolve()
    if not src.is_dir():
        raise NotADirectoryError(f"Not a directory: {src}")

    out = Path(out_zip).resolve() if out_zip else src.with_suffix(".zip")
    out.parent.mkdir(parents=True, exist_ok=True)

    files = [f for f in _iter_files(src) if f.is_file()]
    total = _folder_size(src)
    print(f"📦 Zipping  {src}")
    print(f"   → {out}    ({len(files)} files, {_human_size(total)})")

    t0 = time.time()
    with zipfile.ZipFile(out, "w", compression=compression, allowZip64=True) as zf:
        for f in tqdm(files, unit="file", desc="zip"):
            zf.write(f, arcname=f.relative_to(src.parent))
    print(f"✓ Done in {time.time() - t0:,.1f}s  →  {_human_size(out.stat().st_size)}")
    return out


def unzip_file(zip_path, out_dir=None) -> Path:
    """Extract a single .zip file into out_dir (defaults to alongside the zip)."""
    z = Path(zip_path).resolve()
    if not z.is_file():
        raise FileNotFoundError(f"Not a file: {z}")

    dest = Path(out_dir).resolve() if out_dir else z.with_suffix("")
    dest.mkdir(parents=True, exist_ok=True)

    print(f"📂 Unzipping {z}")
    print(f"   → {dest}")
    t0 = time.time()
    with zipfile.ZipFile(z, "r") as zf:
        members = zf.infolist()
        for m in tqdm(members, unit="file", desc="unzip"):
            zf.extract(m, dest)
    print(f"✓ Done in {time.time() - t0:,.1f}s  ({len(members)} entries)")
    return dest


# ── LZ4 helpers (uses tar + lz4 frame) ───────────────────────────────────────
def tar_lz4_folder(src_folder, out_path=None, compression_level=4) -> Path:
    """Compress a folder into <name>.tar.lz4 (tar archive, lz4-frame compressed)."""
    src = Path(src_folder).resolve()
    if not src.is_dir():
        raise NotADirectoryError(f"Not a directory: {src}")

    out = Path(out_path).resolve() if out_path else src.with_suffix(".tar.lz4")
    out.parent.mkdir(parents=True, exist_ok=True)

    files = [f for f in _iter_files(src) if f.is_file()]
    total = _folder_size(src)
    print(f"📦 LZ4-Zipping  {src}")
    print(f"   → {out}    ({len(files)} files, {_human_size(total)})")

    t0 = time.time()
    with lz4.frame.open(
        out,
        mode="wb",
        compression_level=compression_level,
        block_size=lz4.frame.BLOCKSIZE_MAX4MB,
    ) as lz_out, tarfile.open(fileobj=lz_out, mode="w|") as tar:
        for f in tqdm(files, unit="file", desc="tar.lz4"):
            tar.add(f, arcname=f.relative_to(src.parent))
    print(f"✓ Done in {time.time() - t0:,.1f}s  →  {_human_size(out.stat().st_size)}")
    return out


def untar_lz4_file(lz4_path, out_dir=None) -> Path:
    """Extract a .tar.lz4 archive into out_dir."""
    p = Path(lz4_path).resolve()
    if not p.is_file():
        raise FileNotFoundError(f"Not a file: {p}")

    base_name = p.name
    for suffix in (".tar.lz4", ".tlz4", ".lz4"):
        if base_name.endswith(suffix):
            base_name = base_name[: -len(suffix)]
            break
    dest = Path(out_dir).resolve() if out_dir else (p.parent / base_name)
    dest.mkdir(parents=True, exist_ok=True)

    print(f"📂 LZ4-Unzipping {p}")
    print(f"   → {dest}")
    t0 = time.time()
    count = 0
    with lz4.frame.open(p, mode="rb") as lz_in, tarfile.open(fileobj=lz_in, mode="r|") as tar:
        for member in tar:
            tar.extract(member, dest)
            count += 1
    print(f"✓ Done in {time.time() - t0:,.1f}s  ({count} entries)")
    return dest


# ── TAR.GZ helpers ───────────────────────────────────────────────────────────
def _sniff_tar_compression(path: Path) -> str:
    """Return tarfile mode based on magic bytes (handles mislabeled archives)."""
    with open(path, "rb") as fh:
        head = fh.read(6)
    if head.startswith(b"\x1f\x8b"):
        return "r:gz"
    if head.startswith(b"BZh"):
        return "r:bz2"
    if head.startswith(b"\xfd7zXZ\x00"[:6]):
        return "r:xz"
    return "r:"  # plain tar (no compression)


def untar_gz_file(targz_path, out_dir=None) -> Path:
    """Extract a .tar.gz / .tgz / .tar archive into out_dir.

    Auto-detects compression from the file's magic bytes, so mislabeled archives
    (e.g. a plain tar saved with a .tar.gz extension) extract correctly.
    """
    p = Path(targz_path).resolve()
    if not p.is_file():
        raise FileNotFoundError(f"Not a file: {p}")

    base_name = p.name
    for suffix in (".tar.gz", ".tgz", ".tar"):
        if base_name.endswith(suffix):
            base_name = base_name[: -len(suffix)]
            break
    dest = Path(out_dir).resolve() if out_dir else (p.parent / base_name)
    dest.mkdir(parents=True, exist_ok=True)

    mode = _sniff_tar_compression(p)
    label = {"r:gz": "tar.gz", "r:bz2": "tar.bz2", "r:xz": "tar.xz", "r:": "tar"}[mode]

    print(f"📂 Untarring {p}    ({_human_size(p.stat().st_size)}, detected: {label})")
    print(f"   → {dest}")
    t0 = time.time()
    count = 0
    with tarfile.open(p, mode=mode) as tar:
        members = tar.getmembers()
        for member in tqdm(members, unit="file", desc=label):
            tar.extract(member, dest)
            count += 1
    print(f"✓ Done in {time.time() - t0:,.1f}s  ({count} entries)")
    return dest


print("✓ Helpers ready: zip_folder, unzip_file, tar_lz4_folder, untar_lz4_file, untar_gz_file")

✓ Helpers ready: zip_folder, unzip_file, tar_lz4_folder, untar_lz4_file, untar_gz_file


## 3 — ZIP a single folder

Set `SRC_FOLDER` to the folder you want to zip.
Optionally set `OUT_ZIP` (otherwise it will be created next to the source folder).

In [ ]:
SRC_FOLDER = "/path/to/your/folder"
OUT_ZIP    = None

zip_folder(SRC_FOLDER, OUT_ZIP)

## 4 — ZIP every sub-folder inside a parent folder

Each immediate child folder of `PARENT_FOLDER` is zipped separately.
Output zips are written into `OUT_DIR` (defaults to `PARENT_FOLDER`).

In [ ]:
PARENT_FOLDER = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11/fake/ai_image_reddit_feb25/ai_image_reddit_feb25"
OUT_DIR       = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/ai_image_reddit_feb25"

parent = Path(PARENT_FOLDER).resolve()
out_dir = Path(OUT_DIR).resolve() if OUT_DIR else parent
out_dir.mkdir(parents=True, exist_ok=True)

subdirs = [d for d in parent.iterdir() if d.is_dir()]
print(f"Found {len(subdirs)} sub-folder(s) inside {parent}\n")

for sub in subdirs:
    target_zip = out_dir / f"{sub.name}.zip"
    zip_folder(sub, target_zip)
    print()

print("✓ All sub-folders zipped")

## 5 — UNZIP a single `.zip` file

Set `ZIP_PATH` to the archive you want to extract.
Optionally set `OUT_DIR` (defaults to a folder with the same name next to the zip).

In [4]:
ZIP_PATH = "/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/01.raw/real/unsplash_textures-patterns.zip"
OUT_DIR  = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/real/real_image_website_collection_dec25"

unzip_file(ZIP_PATH, OUT_DIR)

📂 Unzipping /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/01.raw/real/unsplash_textures-patterns.zip
   → /home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/real/real_image_website_collection_dec25


unzip:   0%|          | 0/4875 [00:00<?, ?file/s]

unzip: 100%|██████████| 4875/4875 [02:04<00:00, 39.23file/s] 

✓ Done in 124.4s  (4875 entries)


PosixPath('/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/real/real_image_website_collection_dec25')

## 6 — UNZIP every `.zip` inside a folder

Each `*.zip` directly inside `ZIP_DIR` is extracted into its own sub-folder under `OUT_DIR`.

In [ ]:
ZIP_DIR = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11/fake/ai_image_x_collection_feb25_cleaned"
OUT_DIR = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/fake/ai_image_x_collection_feb25_cleaned/"

zip_dir = Path(ZIP_DIR).resolve()
out_dir = Path(OUT_DIR).resolve() if OUT_DIR else zip_dir
out_dir.mkdir(parents=True, exist_ok=True)

zips = sorted(zip_dir.glob("*.zip"))
print(f"Found {len(zips)} .zip file(s) inside {zip_dir}\n")

for z in zips:
    target = out_dir / z.stem
    unzip_file(z, target)
    print()

print("✓ All .zip files extracted")

## 6.a — UNZIP every `.zip` found **recursively** (preserve folder structure)

Recursively scans `ZIP_ROOT` for every `*.zip` (at any depth) and extracts each one
**in-place** — i.e. into a sibling folder next to the zip, mirroring the original
folder structure exactly.

Options:
- `OUT_ROOT` — if set, the extracted output is mirrored under this root instead
  of next to the source zips (input tree is left untouched).
- `DELETE_ZIP_AFTER` — if `True`, removes each `.zip` after a successful extract.
- `SKIP_IF_EXISTS` — if `True`, skips a zip whose destination folder already exists.

In [ ]:
ZIP_ROOT         = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11/fake/ai_image_x_collection_feb25_cleaned"
OUT_ROOT         = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/fake/ai_image_x_collection_feb25_cleaned"
DELETE_ZIP_AFTER = False
SKIP_IF_EXISTS   = True

src_root = Path(ZIP_ROOT).resolve()
dst_root = Path(OUT_ROOT).resolve() if OUT_ROOT else src_root
dst_root.mkdir(parents=True, exist_ok=True)

zips = sorted(src_root.rglob("*.zip"))
print(f"Found {len(zips)} .zip file(s) recursively under {src_root}")
if OUT_ROOT:
    print(f"Mirroring extracted output under: {dst_root}")
print()

ok = skipped = failed = 0
for z in zips:
    rel_dir   = z.parent.relative_to(src_root)
    target    = (dst_root / rel_dir / z.stem).resolve()
    rel_label = z.relative_to(src_root)

    if SKIP_IF_EXISTS and target.exists() and any(target.iterdir()):
        print(f"⏭  skip (exists): {rel_label}  →  {target.relative_to(dst_root)}")
        skipped += 1
        continue

    try:
        unzip_file(z, target)
        ok += 1
        if DELETE_ZIP_AFTER:
            z.unlink()
            print(f"🗑  removed source zip: {rel_label}")
    except Exception as e:
        failed += 1
        print(f"✗ FAILED {rel_label}: {e}")
    print()

print(f"✓ Done — extracted: {ok}, skipped: {skipped}, failed: {failed}")

## 7 — LZ4 — Compress a single folder (`.tar.lz4`)

Set `SRC_FOLDER` to the folder you want to compress.
Optionally set `OUT_PATH` (defaults to `<src>.tar.lz4` next to the folder).

In [4]:
SRC_FOLDER        = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini"
OUT_PATH          = "/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11-mini.tar.lz4"
COMPRESSION_LEVEL = 1

tar_lz4_folder(SRC_FOLDER, OUT_PATH, compression_level=COMPRESSION_LEVEL)

📦 LZ4-Zipping  /home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini
   → /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11-mini.tar.lz4    (10000 files, 3.06 GB)


tar.lz4: 100%|██████████| 10000/10000 [00:41<00:00, 241.01file/s]


✓ Done in 41.5s  →  3.05 GB


PosixPath('/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11-mini.tar.lz4')

## 8 — LZ4 — Compress every sub-folder inside a parent folder

Each immediate child folder of `PARENT_FOLDER` becomes its own `.tar.lz4` archive.

In [5]:
PARENT_FOLDER     = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini"
OUT_DIR           = "/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11-mini/"
COMPRESSION_LEVEL = 1

parent = Path(PARENT_FOLDER).resolve()
out_dir = Path(OUT_DIR).resolve() if OUT_DIR else parent
out_dir.mkdir(parents=True, exist_ok=True)

subdirs = [d for d in parent.iterdir() if d.is_dir()]
print(f"Found {len(subdirs)} sub-folder(s) inside {parent}\n")

for sub in subdirs:
    target = out_dir / f"{sub.name}.tar.lz4"
    tar_lz4_folder(sub, target, compression_level=COMPRESSION_LEVEL)
    print()

print("✓ All sub-folders compressed to .tar.lz4")

Found 2 sub-folder(s) inside /home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini

📦 LZ4-Zipping  /home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini/v9
   → /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11-mini/v9.tar.lz4    (5000 files, 1.05 GB)


tar.lz4: 100%|██████████| 5000/5000 [00:14<00:00, 347.03file/s]


✓ Done in 14.4s  →  1.04 GB

📦 LZ4-Zipping  /home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini/v11-processing
   → /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11-mini/v11-processing.tar.lz4    (5000 files, 2.02 GB)


tar.lz4: 100%|██████████| 5000/5000 [00:21<00:00, 234.15file/s]

✓ Done in 21.4s  →  2.01 GB

✓ All sub-folders compressed to .tar.lz4


## 9 — LZ4 — Extract a single `.tar.lz4` archive

Set `LZ4_PATH` to the archive you want to extract.
Optionally set `OUT_DIR` (defaults to a folder with the same name next to the archive).

In [13]:
LZ4_PATH = "/home/taiaburrahman/dataset_manager_pro/data/processed/v9/real/dalle_rec_Real.tar.lz4"
OUT_DIR  = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/real/PubDB_Real/"

untar_lz4_file(LZ4_PATH, OUT_DIR)

📂 LZ4-Unzipping /home/taiaburrahman/dataset_manager_pro/data/processed/v9/real/dalle_rec_Real.tar.lz4
   → /home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/real/PubDB_Real
✓ Done in 4.6s  (3105 entries)


PosixPath('/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing/real/PubDB_Real')

## 10 — LZ4 — Extract every `.tar.lz4` inside a folder

Each `*.tar.lz4` directly inside `LZ4_DIR` is extracted into its own sub-folder under `OUT_DIR`.

In [ ]:
LZ4_DIR = "/path/to/folder/with/tar.lz4"
OUT_DIR = None

lz4_dir = Path(LZ4_DIR).resolve()
out_dir = Path(OUT_DIR).resolve() if OUT_DIR else lz4_dir
out_dir.mkdir(parents=True, exist_ok=True)

archives = sorted(lz4_dir.glob("*.tar.lz4"))
print(f"Found {len(archives)} .tar.lz4 file(s) inside {lz4_dir}\n")

for a in archives:
    target = out_dir / a.name[: -len(".tar.lz4")]
    untar_lz4_file(a, target)
    print()

print("✓ All .tar.lz4 archives extracted")

## 11 — TAR.GZ — Extract a single `.tar.gz` archive

Set `TARGZ_PATH` to the archive you want to extract.
Optionally set `OUT_DIR` (defaults to a folder with the same name next to the archive).

In [4]:
TARGZ_PATH = "/home/taiaburrahman/dataset_manager_pro/data/processed/v9/Gen_samples_Grok2_200325.tar.gz"
OUT_DIR    = "/home/taiaburrahman/dataset_manager_pro/data/processed/v9/fake/Gen_samples_Grok2_200325"

untar_gz_file(TARGZ_PATH, OUT_DIR)

📂 Untarring /home/taiaburrahman/dataset_manager_pro/data/processed/v9/Gen_samples_Grok2_200325.tar.gz    (386.25 MB, detected: tar)
   → /home/taiaburrahman/dataset_manager_pro/data/processed/v9/fake/Gen_samples_Grok2_200325


tar: 100%|██████████| 4121/4121 [00:03<00:00, 1346.41file/s]

✓ Done in 4.9s  (4121 entries)


PosixPath('/home/taiaburrahman/dataset_manager_pro/data/processed/v9/fake/Gen_samples_Grok2_200325')

## 12 — TAR.GZ — Extract every `.tar.gz` inside a folder

Each `*.tar.gz` (and `*.tgz`) directly inside `TARGZ_DIR` is extracted into its own
sub-folder under `OUT_DIR` (defaults to `TARGZ_DIR`).

In [ ]:
TARGZ_DIR = "/path/to/folder/with/tar.gz"
OUT_DIR   = None

targz_dir = Path(TARGZ_DIR).resolve()
out_dir   = Path(OUT_DIR).resolve() if OUT_DIR else targz_dir
out_dir.mkdir(parents=True, exist_ok=True)

archives = sorted(
    list(targz_dir.glob("*.tar.gz"))
    + list(targz_dir.glob("*.tgz"))
    + list(targz_dir.glob("*.tar"))
)
print(f"Found {len(archives)} .tar.gz/.tgz/.tar file(s) inside {targz_dir}\n")

for a in archives:
    name = a.name
    for suffix in (".tar.gz", ".tgz", ".tar"):
        if name.endswith(suffix):
            stem = name[: -len(suffix)]
            break
    target = out_dir / stem
    untar_gz_file(a, target)
    print()

print("✓ All tar archives extracted")